# NARR Check & Download — 1985–2026

Verifica e scarica i file NARR raw (`air.2m` e `apcp`) per tutti gli anni **1985–2026**  
usati dal pipeline U-Net SPORTLIS esteso (dominio Northwestern, 484×698 @ 3 km).

**Cosa fa questo notebook:**
1. **Proba PSL NOAA** → scopre fino a che data è disponibile NARR per ogni anno  
2. **Check locale** → verifica che i file `.nc` già scaricati esistano, abbiano dimensione > 0 e siano leggibili  
3. **Scarica** i file mancanti o corrotti via OPeNDAP (`psl.noaa.gov/thredds/dodsC`)  
4. **Report finale** con tabella riassuntiva per anno

**Note NARR:**
- NARR copre dal 1979 in poi, aggiornato con ~1-2 settimane di ritardo rispetto a real-time  
- 2026 sarà parziale (es. Jan–Apr/May a seconda del momento in cui si esegue)  
- Risoluzione temporale: 3h → 2920 step/anno (anni normali), 2928 bisestili

**Output:** `NARR_RAW_DIR/narr_ext_{tair|precip}_{year}.nc`

In [ ]:
# ── CONFIGURAZIONE ────────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc4
import warnings, time, traceback
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Path NCAR/Derecho ─────────────────────────────────────────────────────────
NCAR_HOME    = Path('/glade/u/home/cionni')
NCAR_SCRATCH = NCAR_HOME / 'derecho_scratch'
PROJECT_DIR  = NCAR_HOME / 'unet_sportlis'
NARR_RAW_DIR = NCAR_SCRATCH / 'narr_extended_raw'
NARR_RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── NARR OPeNDAP ──────────────────────────────────────────────────────────────
NARR_BASE   = 'https://psl.noaa.gov/thredds/dodsC/Datasets/NARR/monolevel'
NARR_VARS   = {'air.2m': 'tair', 'apcp': 'precip'}   # prefix → out_name
NARR_OPEN_KW = dict(decode_times=True, chunks={'time': 240})

# ── Bbox dominio esteso ───────────────────────────────────────────────────────
NARR_BBOX = dict(lat_min=34.5, lat_max=50.0, lon_min=-126.0, lon_max=-103.5)

# ── Anni da gestire ───────────────────────────────────────────────────────────
HIST_YEARS  = list(range(1985, 2025))   # anni di training (+ 1984 come buffer)
PROJ_YEARS  = [2025, 2026]              # anni di proiezione
ALL_YEARS   = list(range(1984, 2027))   # 1984 serve come buffer per 2025 (rolling 60d)

# ── Soglia minima file (bytes) ────────────────────────────────────────────────
MIN_FILE_BYTES = 5 * 1024 * 1024        # 5 MB (un anno NARR subset pesa ~40-300 MB)
MIN_TIMESTEPS  = 200                    # almeno 200 timestep per considerare un file valido

# ── Retry & timeout ───────────────────────────────────────────────────────────
N_RETRIES      = 3
RETRY_WAIT_S   = 30
OVERWRITE      = False   # True = ri-scarica anche se il file esiste

print(f'NARR_RAW_DIR : {NARR_RAW_DIR}')
print(f'NARR_RAW_DIR exists: {NARR_RAW_DIR.exists()}')
print(f'Anni totali da gestire: {ALL_YEARS[0]}–{ALL_YEARS[-1]}  ({len(ALL_YEARS)} anni)')
print(f'Timestamp esecuzione: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

In [ ]:
# ── CELLA 1: PROBA NARR PSL — fino a dove è disponibile ogni anno ──────────────
# Per ogni anno prova ad aprire il file via OPeNDAP e registra n_timestep e
# data finale. Per anni storici (≤2024) basta verificare che il file esista;
# per 2025-2026 serve sapere fino a dove arriva.

def probe_narr_year(year, prefix, timeout_s=60):
    """
    Apre il file NARR via OPeNDAP e restituisce:
      (n_timesteps, first_time, last_time)  oppure  None se non disponibile.
    Non scarica nulla, legge solo i metadati (variabile 'time').
    """
    url = f'{NARR_BASE}/{prefix}.{year}.nc'
    try:
        # Usa netCDF4 direttamente per leggere solo la variabile time
        ds = nc4.Dataset(url)
        t_var = ds.variables['time']
        n = len(t_var)
        times = nc4.num2date(t_var[:2], units=t_var.units,
                             calendar=getattr(t_var, 'calendar', 'standard'))
        times_end = nc4.num2date(t_var[-2:], units=t_var.units,
                                 calendar=getattr(t_var, 'calendar', 'standard'))
        t0 = pd.Timestamp(str(times[0]))
        t1 = pd.Timestamp(str(times_end[-1]))
        ds.close()
        return n, t0, t1
    except Exception as e:
        return None, None, str(e)[:80]


print('Probing NARR PSL per anni 2023–2026 (per anni storici non serve probe completo)...')
print(f'{"Anno":<6} {"Variabile":<10} {"N timestep":>12}  {"Inizio":<22} {"Fine":<22}  Stato')
print('-'*90)

narr_availability = {}   # year -> {prefix: (n, t0, t1)}

# Proba estesa solo per anni recenti (storici assumiamo disponibili)
probe_years = list(range(2022, 2027))

for year in probe_years:
    narr_availability[year] = {}
    for prefix in NARR_VARS:
        n, t0, t1 = probe_narr_year(year, prefix)
        narr_availability[year][prefix] = (n, t0, t1)
        if n is not None:
            print(f'{year:<6} {prefix:<10} {n:>12}  {str(t0):<22} {str(t1):<22}  OK')
        else:
            print(f'{year:<6} {prefix:<10} {"N/A":>12}  {"":<22} {"":<22}  ERRORE: {t1}')

# Anni storici: assumiamo disponibili, verifichiamo solo localmente
for year in ALL_YEARS:
    if year not in narr_availability:
        narr_availability[year] = {p: ('assumed_ok', None, None) for p in NARR_VARS}

print('\nProbe completato.')

In [ ]:
# ── CELLA 2: CALCOLA SLICES BBOX (una-tantum) ─────────────────────────────────
# Legge le coordinate NARR da un anno di riferimento e calcola gli indici
# y/x che corrispondono al bbox del dominio esteso.

def get_narr_yx_slices(ds, bbox):
    """Trova gli indici y,x NARR che coprono il bbox."""
    if 'lat' in ds.coords and 'lon' in ds.coords:
        lat2d = ds['lat'].values
        lon2d = ds['lon'].values
    else:
        # Cerca variabile lat/lon tra data_vars o coords
        for cand in ['lat','latitude','LAT']:
            if cand in ds:
                lat2d = ds[cand].values; break
        for cand in ['lon','longitude','LON']:
            if cand in ds:
                lon2d = ds[cand].values; break

    # Converti longitudine NARR (0-360) in (-180,180)
    lon2d = np.where(lon2d > 180, lon2d - 360, lon2d)

    mask = ((lat2d >= bbox['lat_min']) & (lat2d <= bbox['lat_max']) &
            (lon2d >= bbox['lon_min']) & (lon2d <= bbox['lon_max']))
    rows, cols = np.where(mask)
    if len(rows) == 0:
        raise ValueError('Nessuna cella NARR nel bbox — controlla coordinate.')
    y0, y1 = rows.min(), rows.max() + 1
    x0, x1 = cols.min(), cols.max() + 1
    return (slice(y0, y1), slice(x0, x1)), lat2d[y0:y1, x0:x1], lon2d[y0:y1, x0:x1]


print('Calcolo slices bbox da anno di riferimento (2020)...')
REF_YEAR = 2020
ref_url = f'{NARR_BASE}/air.2m.{REF_YEAR}.nc'
print(f'  URL: {ref_url}')

ds_ref = xr.open_dataset(ref_url, **NARR_OPEN_KW)
(YS, XS), NARR_LAT2D, NARR_LON2D = get_narr_yx_slices(ds_ref, NARR_BBOX)
ds_ref.close()

print(f'  Bbox slices: y={YS}, x={XS}')
print(f'  NARR subset shape: {NARR_LAT2D.shape}')
print(f'  lat range: {NARR_LAT2D.min():.2f}–{NARR_LAT2D.max():.2f}')
print(f'  lon range: {NARR_LON2D.min():.2f}–{NARR_LON2D.max():.2f}')

In [ ]:
# ── CELLA 3: FUNZIONI CHECK E DOWNLOAD ────────────────────────────────────────

def local_file(year, out_name):
    return NARR_RAW_DIR / f'narr_ext_{out_name}_{year}.nc'


def check_local_file(year, out_name):
    """
    Verifica il file locale. Restituisce dict con:
      exists, size_mb, readable, n_timesteps, t0, t1, ok, reason
    """
    fpath = local_file(year, out_name)
    result = dict(path=fpath, exists=False, size_mb=0.0,
                  readable=False, n_timesteps=0, t0=None, t1=None,
                  ok=False, reason='')

    if not fpath.exists():
        result['reason'] = 'file non esiste'
        return result

    result['exists'] = True
    size = fpath.stat().st_size
    result['size_mb'] = size / 1e6

    if size < MIN_FILE_BYTES:
        result['reason'] = f'file troppo piccolo ({size/1e6:.1f} MB < {MIN_FILE_BYTES/1e6:.0f} MB)'
        return result

    # Tenta lettura
    try:
        ds = xr.open_dataset(str(fpath), chunks={'time': 240})
        n = len(ds['time'])
        result['n_timesteps'] = n
        result['t0'] = pd.Timestamp(str(ds['time'].values[0]))
        result['t1'] = pd.Timestamp(str(ds['time'].values[-1]))
        ds.close()
        result['readable'] = True
        if n < MIN_TIMESTEPS:
            result['reason'] = f'troppo pochi timestep ({n} < {MIN_TIMESTEPS})'
            return result
        result['ok'] = True
        result['reason'] = 'OK'
    except Exception as e:
        result['reason'] = f'errore lettura: {str(e)[:80]}'

    return result


def download_narr_year(year, prefix, out_name, yx_slices, overwrite=False):
    """
    Scarica subset NARR via OPeNDAP e salva netCDF locale.
    yx_slices = (slice_y, slice_x).
    Gestisce anni parziali (es. 2026): salva ciò che è disponibile.
    Ritorna (True, n_timesteps) se ok, (False, motivo) se fallisce.
    """
    out_file = local_file(year, out_name)
    if out_file.exists() and not overwrite:
        return True, f'già presente ({out_file.stat().st_size/1e6:.0f} MB)'

    url = f'{NARR_BASE}/{prefix}.{year}.nc'
    ys, xs = yx_slices

    for attempt in range(1, N_RETRIES + 1):
        try:
            print(f'    [{attempt}/{N_RETRIES}] Apro {url} ...')
            ds = xr.open_dataset(url, **NARR_OPEN_KW)

            # Trova la variabile dati principale
            skip = {'lat','lon','latitude','longitude','lambert_conformal',
                    'reftime','forecast_reference_time'}
            data_vars = [v for v in ds.data_vars if v.lower() not in skip]
            if not data_vars:
                ds.close()
                return False, 'nessuna variabile dati trovata'
            var_name = data_vars[0]

            n_t = len(ds['time'])
            t0 = pd.Timestamp(str(ds['time'].values[0]))
            t1 = pd.Timestamp(str(ds['time'].values[-1]))
            print(f'    variabile: {var_name}  timestep: {n_t}  {t0.date()} → {t1.date()}')

            # Subset spaziale + load in memoria a blocchi
            # Dimensioni: (time, y, x) oppure (time, x, y) a seconda della versione
            da = ds[var_name]
            # Gestisci ordine dimensioni
            dims = da.dims
            if 'y' in dims and 'x' in dims:
                da_sub = da.isel(y=ys, x=xs)
            elif 'lat' in dims and 'lon' in dims:
                da_sub = da.isel(lat=ys, lon=xs)
            else:
                # fallback: assumi (time, dim1, dim2)
                d1, d2 = dims[1], dims[2]
                da_sub = da.isel({d1: ys, d2: xs})

            # Scarica a blocchi e salva
            print(f'    Scarico e salvo → {out_file.name} ...')
            ds_out = da_sub.to_dataset(name=var_name)

            # Aggiungi lat/lon 2D se presenti
            for coord_name in ['lat','lon','latitude','longitude']:
                if coord_name in ds.coords or coord_name in ds:
                    try:
                        coord_da = (ds[coord_name] if coord_name in ds
                                    else ds.coords[coord_name])
                        c_dims = coord_da.dims
                        if len(c_dims) == 2:
                            cd1, cd2 = c_dims
                            if cd1 in ('y','lat'): coord_sub = coord_da.isel({cd1:ys, cd2:xs})
                            else: coord_sub = coord_da.isel({cd1:xs, cd2:ys})
                            ds_out[coord_name] = coord_sub
                    except Exception:
                        pass

            ds.close()

            # Salva con compressione
            enc = {var_name: {'zlib': True, 'complevel': 4, 'dtype': 'float32'}}
            ds_out.to_netcdf(str(out_file), encoding=enc)
            size_mb = out_file.stat().st_size / 1e6
            print(f'    Salvato: {out_file.name}  ({size_mb:.1f} MB)')
            return True, n_t

        except Exception as e:
            err = traceback.format_exc()
            print(f'    ERRORE attempt {attempt}: {str(e)[:120]}')
            if attempt < N_RETRIES:
                print(f'    Attendo {RETRY_WAIT_S}s prima di riprovare...')
                time.sleep(RETRY_WAIT_S)
            else:
                return False, str(e)[:120]

    return False, 'esauriti i tentativi'


print('Funzioni check e download definite.')

In [ ]:
# ── CELLA 4: CHECK TUTTI I FILE LOCALI ────────────────────────────────────────
# Produce una tabella con lo stato di ogni anno × variabile.

print(f'Checking {len(ALL_YEARS)} anni × {len(NARR_VARS)} variabili = '
      f'{len(ALL_YEARS)*len(NARR_VARS)} file...\n')

check_results = {}   # (year, out_name) → dict

for year in ALL_YEARS:
    for prefix, out_name in NARR_VARS.items():
        r = check_local_file(year, out_name)
        check_results[(year, out_name)] = r

# ── Stampa tabella ─────────────────────────────────────────────────────────────
print(f'{"Anno":<6} {"tair":^22} {"precip":^22}  Stato')
print('-'*60)

missing_or_bad = []   # (year, prefix, out_name)

for year in ALL_YEARS:
    row_ok = []
    for prefix, out_name in NARR_VARS.items():
        r = check_results[(year, out_name)]
        row_ok.append(r['ok'])
        if not r['ok']:
            missing_or_bad.append((year, prefix, out_name))

    # Stampa solo anni con almeno un problema (o anni recenti sempre)
    all_ok = all(row_ok)
    always_show = year >= 2023
    if not all_ok or always_show:
        parts = []
        for prefix, out_name in NARR_VARS.items():
            r = check_results[(year, out_name)]
            if r['ok']:
                s = f'OK {r["n_timesteps"]}ts {r["size_mb"]:.0f}MB'
            else:
                s = f'✗  {r["reason"][:18]}'
            parts.append(f'{s:<22}')
        status = '✓ OK' if all_ok else '⚠ PROBLEMA'
        print(f'{year:<6} {" ".join(parts)}  {status}')

n_missing = len(missing_or_bad)
print(f'\n--- SOMMARIO ---')
print(f'File mancanti o corrotti: {n_missing}')
if missing_or_bad:
    print('Lista:')
    for year, prefix, out_name in missing_or_bad:
        r = check_results[(year, out_name)]
        print(f'  {year}  narr_ext_{out_name}_{year}.nc  → {r["reason"]}')

In [ ]:
# ── CELLA 5: DOWNLOAD FILE MANCANTI ───────────────────────────────────────────
# Scarica tutti i file mancanti o corrotti identificati nella cella precedente.
# Per 2026: scarica ciò che è disponibile (anno parziale).
# Imposta OVERWRITE = True in cfg per forzare il re-download.

# Inizializza slices (se non già fatto, ad es. se si salta la cella 2)
try:
    _ = YS
except NameError:
    print('Ricalcolo YX_SLICES...')
    ref_url = f'{NARR_BASE}/air.2m.2020.nc'
    ds_ref = xr.open_dataset(ref_url, **NARR_OPEN_KW)
    (YS, XS), NARR_LAT2D, NARR_LON2D = get_narr_yx_slices(ds_ref, NARR_BBOX)
    ds_ref.close()
    print(f'  Slices: y={YS}, x={XS}')

yx_slices = (YS, XS)

if not missing_or_bad and not OVERWRITE:
    print('Nessun file da scaricare. Tutti i file sono presenti e validi.')
else:
    # Lista da scaricare: mancanti + (opzionale) tutti se OVERWRITE
    to_download = missing_or_bad if not OVERWRITE else [
        (y, p, n) for y in ALL_YEARS for p, n in NARR_VARS.items()
    ]

    # Rimuovi anni per cui NARR non è disponibile affatto
    to_download_filtered = []
    for year, prefix, out_name in to_download:
        avail = narr_availability.get(year, {}).get(prefix, ('assumed_ok', None, None))
        if avail[0] is None:
            print(f'  {year} {out_name}: SKIP — NARR non disponibile su PSL ({avail[2]})')
        else:
            to_download_filtered.append((year, prefix, out_name))

    print(f'File da scaricare: {len(to_download_filtered)}')
    print('='*60)

    download_log = []
    for i, (year, prefix, out_name) in enumerate(to_download_filtered, 1):
        print(f'\n[{i}/{len(to_download_filtered)}] {year}  {prefix} → narr_ext_{out_name}_{year}.nc')
        t_start = time.time()
        ok, info = download_narr_year(year, prefix, out_name, yx_slices, overwrite=OVERWRITE)
        elapsed = time.time() - t_start
        status_str = 'OK' if ok else 'FAIL'
        print(f'  → {status_str}  {info}  ({elapsed:.0f}s)')
        download_log.append(dict(year=year, out_name=out_name, ok=ok,
                                 info=str(info), elapsed_s=round(elapsed)))

    print('\n' + '='*60)
    print(f'Download completato: {sum(d["ok"] for d in download_log)}/{len(download_log)} OK')
    failed = [d for d in download_log if not d['ok']]
    if failed:
        print('\nFalliti:')
        for d in failed:
            print(f'  {d["year"]} {d["out_name"]}: {d["info"]}')

In [ ]:
# ── CELLA 6: RE-CHECK DOPO DOWNLOAD ──────────────────────────────────────────
# Ri-verifica tutti i file e stampa il report finale completo.

print('Re-check dopo download...\n')

rows = []
for year in ALL_YEARS:
    row = {'year': year}
    for prefix, out_name in NARR_VARS.items():
        r = check_local_file(year, out_name)
        row[f'{out_name}_ok']    = r['ok']
        row[f'{out_name}_mb']    = round(r['size_mb'], 1)
        row[f'{out_name}_nts']   = r['n_timesteps']
        row[f'{out_name}_t0']    = str(r['t0'])[:10] if r['t0'] else ''
        row[f'{out_name}_t1']    = str(r['t1'])[:10] if r['t1'] else ''
        row[f'{out_name}_reason'] = r['reason']
    rows.append(row)

df = pd.DataFrame(rows)

# ── Stampa tabella riassuntiva ─────────────────────────────────────────────────
print(f'{"Anno":<6} {"tair ts":>8} {"tair t1":<12} {"tair MB":>8}  '
      f'{"precip ts":>10} {"precip t1":<12} {"precip MB":>10}  Stato')
print('-'*85)

for _, r in df.iterrows():
    y = int(r['year'])
    t_ok  = r['tair_ok']
    p_ok  = r['precip_ok']
    stato = '✓' if (t_ok and p_ok) else '✗'

    # Mostra sempre anni recenti; per anni storici ok mostra solo sommario
    if (not t_ok or not p_ok) or y >= 2023:
        print(f'{y:<6} '
              f'{r["tair_nts"]:>8} {r["tair_t1"]:<12} {r["tair_mb"]:>8.1f}  '
              f'{r["precip_nts"]:>10} {r["precip_t1"]:<12} {r["precip_mb"]:>10.1f}  {stato}')
        if not t_ok:
            print(f'       tair   → {r["tair_reason"]}')
        if not p_ok:
            print(f'       precip → {r["precip_reason"]}')

# Anni storici ok: stampa solo conteggio
hist_ok = df[(df['year'] < 2023) & df['tair_ok'] & df['precip_ok']]
if len(hist_ok):
    print(f'... [{len(hist_ok)} anni {int(hist_ok.year.min())}–{int(hist_ok.year.max())} tutti OK] ...')

# ── Sommario finale ───────────────────────────────────────────────────────────
n_ok    = int((df['tair_ok'] & df['precip_ok']).sum())
n_bad   = len(df) - n_ok
bad_df  = df[~(df['tair_ok'] & df['precip_ok'])]

print('\n' + '='*50)
print(f'TOTALE anni: {len(ALL_YEARS)}')
print(f'  Completi (tair + precip OK): {n_ok}')
print(f'  Con problemi:                {n_bad}')
if n_bad:
    print(f'  Anni con problemi: {sorted(bad_df.year.astype(int).tolist())}')
else:
    print('  ✓ Tutti i file sono presenti e validi.')

# Mostra disponibilità 2026
r26 = df[df['year'] == 2026]
if not r26.empty:
    t1_2026 = r26['tair_t1'].values[0]
    nts_2026 = r26['tair_nts'].values[0]
    print(f'\n2026 NARR disponibile fino a: {t1_2026}  ({nts_2026} timestep = '
          f'{int(nts_2026)//8} giorni)')

print(f'\nReport generato: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

In [ ]:
# ── CELLA 7 (opzionale): ESPORTA REPORT CSV ──────────────────────────────────

report_path = NARR_RAW_DIR / 'narr_check_report.csv'
df.to_csv(str(report_path), index=False)
print(f'Report salvato: {report_path}')
print(df[['year','tair_ok','tair_nts','tair_t1','tair_mb',
          'precip_ok','precip_nts','precip_t1','precip_mb']].to_string(index=False))